# HW1: After Dark at Berkeley

**COMPSS 211A | Fall 2026 | Student copy**

Use a synthetic late-night campus commute survey to demonstrate independent Python and Pandas fundamentals.

**Date/deadline:** Sunday, September 27, 2026 at 11:59 p.m.

Complete every required function and written response. Keep the named outputs so the notebook checks can find them.

## Scenario

The fictional Berkeley Methods Studio needs a short descriptive brief on how graduate students get home after evening classes. The survey is synthetic. Use it to practice the analysis without making claims about actual Berkeley students.

## What you will practice

- Audit shape, types, missingness, and category values.
- Create useful filters, grouped summaries, and derived columns.
- Distinguish zero, missing, and not-applicable values.
- Interpret descriptive results without causal overclaiming.

## Keep handy

- **Selection:** Use `.loc[condition, columns]` and `.copy()` for auditable subsets.
- **Grouping:** Use named aggregation so every output column has an interpretable name.
- **Missingness:** Do not replace missing commute values with zero unless zero is substantively correct.
- **Documentation:** Pandas user guide: https://pandas.pydata.org/docs/user_guide/

## AI and collaboration policy

No generative AI tools are allowed. Use course notes, documentation, office hours, and concept-level peer discussion.

In [19]:
from pathlib import Path
import json
import os
import sys

import pandas as pd
import numpy as np
from IPython.display import display

SUPPORTED_PYTHON = (3, 13)
if sys.version_info[:2] != SUPPORTED_PYTHON:
    print(
        "Setup note: this course is tested with Python 3.13; "
        f"you are running {sys.version.split()[0]}."
    )

def find_course_root():
    """Find the cloned repository when this notebook is running locally."""
    for folder in (Path.cwd(), *Path.cwd().parents):
        if (folder / "pyproject.toml").exists() and (folder / "data").is_dir():
            return folder
    return None

LOCAL_COURSE_ROOT = find_course_root()
COURSE_ROOT = LOCAL_COURSE_ROOT or Path.cwd()
DATA_DIR = (
    LOCAL_COURSE_ROOT / "data"
    if LOCAL_COURSE_ROOT
    else COURSE_ROOT / "compss211_data"
)
GENERATED_DIR = COURSE_ROOT / "generated"
DATA_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_DIR.mkdir(parents=True, exist_ok=True)

DATA_BASE_URL = (
    "https://raw.githubusercontent.com/"
    "macss-berkeley/compss-211a/main/data"
)

def course_data_path(filename):
    """Use a local course file, or download it when running in Colab."""
    path = DATA_DIR / filename
    if not path.exists():
        from urllib.request import urlretrieve

        urlretrieve(f"{DATA_BASE_URL}/{filename}", path)
    return path

print(f"Python {sys.version.split()[0]} | data={DATA_DIR}")

Python 3.13.15 | data=/Users/vynguyen/compss-211a/data


## 1. Audit the survey

Implement `audit_commute_data(frame)`.

**Required output:** a nine-row DataFrame with columns `column`,
`dtype`, `missing_count`, `missing_share`, and `unique_non_null`.
For a toy column `[1, None, 1]`, the missing count is 1 and the
non-null unique count is 1.

In [20]:
survey = pd.read_csv(course_data_path("hw1_after_dark_commute_survey.csv"))
display(survey.head())

,respondent_id,program,commute_mode,commute_minutes_one_way,days_on_campus,leaves_after_8pm_days,monthly_transport_cost_usd,wellbeing_score,reliable_internet
0,BMS-001,Information,Walk,18.0,5,2,0.00,6.8,Yes
1,BMS-002,Public Policy,Walk,13.0,4,2,0.00,6.2,Yes
2,BMS-003,Public Policy,Bike,16.0,5,1,92.27,7.5,Yes
3,BMS-004,Social Welfare,Remote,0.0,0,0,0.00,7.1,Yes
4,BMS-005,Public Policy,Walk,25.0,3,0,0.00,7.3,Yes


In [ ]:
def audit_commute_data(frame): # defines a function to audit_commute_data, which accepts DataFrame parameter called frame
    """Return one audit row for every column in frame.

    Required columns: column, dtype, missing_count, missing_share,
    unique_non_null. Preserve the source column order.
    """
    
    nrows = len(frame) # number of rows in the DataFrame
    audit_row = [] # creates empty list to store audit rows

    for col in frame.columns: #starts a for loop to iterate over each column in the DataFrame
        s = frame[col] # selects the column as a Series
        missing_count = s.isna().sum() # counts the number of missing values in the column
        missing_share = missing_count / nrows # calculates the proportion of missing values
        unique_non_null = s.nunique() # counts the number of unique non-null values in the column
        audit_row.append({ # appends a dictionary representing the audit row for the current column to the audit_row list
            'column': col, # name of the column
            'dtype': s.dtype, # data type of the column
            'missing_count': missing_count, # number of missing values in the column
            'missing_share': missing_share, # proportion of missing values in the column
            'unique_non_null': unique_non_null # number of unique non-null values in the column
        }) 
    return pd.DataFrame(audit_row) # returns a DataFrame containing the audit rows for all columns in the original frame
    raise NotImplementedError("Implement audit_commute_data") # raises an error if the function is not implemented (should not be reached)

In [22]:
# After implementing the function, create and display audit_table.
audit_table = audit_commute_data(survey) # creates the audit table for the survey DataFrame
audit_table # displays table

,column,dtype,missing_count,missing_share,unique_non_null
0,respondent_id,object,0,0.000000,96
1,program,object,0,0.000000,4
2,commute_mode,object,0,0.000000,6
3,commute_minutes_one_way,float64,3,0.031250,50
4,days_on_campus,int64,0,0.000000,5
5,leaves_after_8pm_days,int64,0,0.000000,5
6,monthly_transport_cost_usd,float64,2,0.020833,63
7,wellbeing_score,float64,0,0.000000,36
8,reliable_internet,object,0,0.000000,2


## 2. Add commute metrics

Implement `add_commute_metrics(frame)` without mutating its input.

**Required output:** `commute_metrics`, containing every source column
plus `weekly_commute_minutes` and `late_night_share`. A row with a
20-minute one-way commute and 4 campus days has 160 weekly minutes.
A remote row with 0 campus days has a late-night share of 0.0.

In [ ]:
def add_commute_metrics(frame):
    """Return a copy with two new columns.

    weekly_commute_minutes:
        2 * one-way minutes * days on campus
    late_night_share:
        late-night days / days on campus; use 0.0 when campus days is 0.
    Do not replace missing commute minutes with zero and do not mutate frame.
    """
    
    commute_metrics = frame.copy() # creates a copy of the original frame
    commute_metrics['weekly_commute_minutes'] = (2 * commute_metrics['commute_minutes_one_way'] * commute_metrics['days_on_campus']) 
    # calculates weekly commute minutes, assuming a round trip each day (hence why we multiply by 2)

    days = commute_metrics['days_on_campus']
    late_night_days = commute_metrics['leaves_after_8pm_days']
    commute_metrics['late_night_share'] = np.where(days == 0, 0.0, late_night_days / days)
    # uses np.where to set late_night_share to 0.0 when days_on_campus is 0, otherwise calculates the share of late-night days
    return commute_metrics

    raise NotImplementedError("Implement add_commute_metrics")

In [26]:
# Create commute_metrics and display its first five rows.
commute_metrics = add_commute_metrics(survey) # creates the commute metrics DataFrame
commute_metrics.head() # displays the first five rows of the commute metrics DataFrame

,respondent_id,program,commute_mode,commute_minutes_one_way,days_on_campus,leaves_after_8pm_days,monthly_transport_cost_usd,wellbeing_score,reliable_internet,weekly_commute_minutes,late_night_share
0,BMS-001,Information,Walk,18.0,5,2,0.00,6.8,Yes,180.0,0.4
1,BMS-002,Public Policy,Walk,13.0,4,2,0.00,6.2,Yes,104.0,0.5
2,BMS-003,Public Policy,Bike,16.0,5,1,92.27,7.5,Yes,160.0,0.2
3,BMS-004,Social Welfare,Remote,0.0,0,0,0.00,7.1,Yes,0.0,0.0
4,BMS-005,Public Policy,Walk,25.0,3,0,0.00,7.3,Yes,150.0,0.0


## 3. Select the after-dark cases

Implement `select_after_dark(frame, min_late_days=2,
min_weekly_minutes=240)`.

**Required output:** `after_dark_cases`, containing only rows meeting
both thresholds and sorted by weekly commute minutes from highest to
lowest. Call the function with its default thresholds.

In [32]:
def select_after_dark(
    frame,
    min_late_days=2,
    min_weekly_minutes=240,
):
    """Return rows meeting both thresholds.

    Sort by weekly_commute_minutes descending, then respondent_id
    ascending. Return a copy rather than a view.
    """

    condition = (
        (frame['weekly_commute_minutes'] >= min_weekly_minutes) & # checks if weekly commute minutes meet the minimum threshold
        (frame['leaves_after_8pm_days'] >= min_late_days) # checks if the number of late-night days meets the minimum threshold
    )
    after_dark_cases = frame.loc[condition].copy() # creates a copy of the rows that meet both thresholds
    after_dark_cases = after_dark_cases.sort_values( # sorts the after_dark_cases by weekly_commute_minutes descending and respondent_id ascending
        by=['weekly_commute_minutes', 'respondent_id'],
        ascending=[False, True]
    )
    return after_dark_cases
    raise NotImplementedError("Implement select_after_dark")

In [33]:
# Create after_dark_cases with the default thresholds and display it.
after_dark_cases = select_after_dark(commute_metrics) # creates the after_dark_cases DataFrame with default thresholds
after_dark_cases

,respondent_id,program,commute_mode,commute_minutes_one_way,days_on_campus,leaves_after_8pm_days,monthly_transport_cost_usd,wellbeing_score,reliable_internet,weekly_commute_minutes,late_night_share
84,BMS-085,MaCSS,BART + walk,68.0,4,2,63.74,5.2,Yes,544.0,0.500000
21,BMS-022,Information,BART + walk,65.0,4,2,113.72,8.3,Yes,520.0,0.500000
7,BMS-008,MaCSS,BART + walk,60.0,4,3,153.10,4.4,Yes,480.0,0.750000
38,BMS-039,Social Welfare,BART + walk,58.0,4,2,118.19,6.6,Yes,464.0,0.500000
9,BMS-010,Information,BART + walk,46.0,5,2,0.00,6.8,Yes,460.0,0.400000
70,BMS-071,Information,AC Transit,45.0,5,3,151.67,4.9,Yes,450.0,0.600000
62,BMS-063,Public Policy,BART + walk,65.0,3,3,63.10,7.2,Yes,390.0,1.000000
63,BMS-064,Information,AC Transit,38.0,5,4,173.04,7.8,Yes,380.0,0.800000
44,BMS-045,Information,AC Transit,62.0,3,3,34.72,6.1,Yes,372.0,1.000000
95,BMS-096,Social Welfare,BART + walk,46.0,4,2,64.98,8.8,No,368.0,0.500000


## 4. Summarize and export

Implement `summarize_by_mode(frame)`.

**Required output:** `mode_summary`, with one row per commute mode and
the six specified columns. Export it to
`generated/hw1_mode_summary.csv` without an
extra index column.

In [36]:
def summarize_by_mode(frame):
    """Return one row per commute_mode.

    Required columns: commute_mode, respondents,
    median_one_way_minutes, mean_weekly_minutes,
    mean_late_night_share, mean_wellbeing.
    Sort by mean_weekly_minutes descending.
    """

    mode_summary = (
        frame.groupby('commute_mode').agg( # aggregates the data by commute_mode, calculating various summary statistics for each mode
            respondents = ('respondent_id', 'count'), # counts the number of respondents for each commute mode
            median_one_way_minutes = ('commute_minutes_one_way', 'median'), # calculates the median one-way commute time for each mode
            mean_weekly_minutes = ('weekly_commute_minutes', 'mean'), # calculates the mean weekly commute time for each mode
            mean_late_night_share = ('late_night_share', 'mean'), # calculates the mean share of late-night commutes for each mode
            mean_wellbeing = ('wellbeing_score', 'mean') # calculates the mean wellbeing score for each mode
        ).reset_index() # resets the index of the resulting DataFrame after grouping
    )

    mode_summary = mode_summary.sort_values(
        by = 'mean_weekly_minutes', # sorts the DataFrame by the mean weekly commute time in descending order
        ascending = False
    )
    return mode_summary
    raise NotImplementedError("Implement summarize_by_mode")

In [37]:
# Create mode_summary, export hw1_mode_summary.csv, and display it.
mode_summary = summarize_by_mode(commute_metrics)  # creates the mode_summary DataFrame by summarizing commute metrics by mode
mode_summary.to_csv('hw1_mode_summary.csv', index=False)  # exports the mode_summary DataFrame to a CSV file
mode_summary

,commute_mode,respondents,median_one_way_minutes,mean_weekly_minutes,mean_late_night_share,mean_wellbeing
1,BART + walk,23,54.0,374.782609,0.553623,6.639130
0,AC Transit,19,42.0,277.473684,0.597368,6.226316
3,Drive,11,32.0,213.800000,0.371212,6.663636
2,Bike,13,24.5,170.666667,0.442308,7.053846
5,Walk,21,18.0,126.200000,0.562698,6.833333
4,Remote,9,0.0,0.000000,0.000000,6.900000


## 5. Interpret the displayed evidence

Write four to six sentences. Cite at least two values from your
outputs, explain one missing-or-zero decision, and state why the
descriptive comparison does not establish causality.

### Your response

**Prompt:** What does the analysis show, and what can it not establish?

> In analyzing the survey, I noticed that those who commute via BART (and walking) tend to have the longest commute--they have the longest median one-way commute in minutes (54) and weekly mean minutes (375, or 6.25 hours). This is compared to those who walk, (median one-way minutes = 18, mean weekly minutes = 126 or about 2.1 hours).

> I also tried to be mindful of missing commute values in the survey, as I am aware that they do not necessarily indicate a zero-minute commute; thus, NaN values remain as NaN and are not converted into zeroes when calculating the late night shares.

> However, while these descriptive comparisons may highlight correlations accross transport modes, they do not established causality because they do not consider confounding variables like location or individual schedules, which may independently impact commute choices or the amount of time spent on campus after 8pm.